# TrialExamples

This notebook is a Python-native tutorial derived from the MATLAB workflow name, implemented from scratch for `nSTAT-python`.

- Execution group: `smoke`
- Workflow family: `signal`
- Paper DOI: `10.1016/j.jneumeth.2012.08.009`
- PMID: `22981419`
- Help page: `docs/help/examples/TrialExamples.md`


Notebook source link: [TrialExamples.ipynb](https://github.com/cajigaslab/nSTAT-python/blob/main/notebooks/TrialExamples.ipynb)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from nstat.analysis import Analysis
from nstat.cif import CIFModel
from nstat.decoding import DecodingAlgorithms
from nstat.events import Events
from nstat.history import HistoryBasis
from nstat.signal import Covariate
from nstat.spikes import SpikeTrain, SpikeTrainCollection
from nstat.trial import CovariateCollection, Trial, TrialConfig

TOPIC = "TrialExamples"
FAMILY = "signal"
rng = np.random.default_rng(2026)
print(f"Running notebook topic: {TOPIC} (family={FAMILY})")

def validate_numeric_checkpoints(metrics: dict[str, float], limits: dict[str, tuple[float, float]], topic: str) -> None:
    if not metrics:
        raise AssertionError(f"TrialExamples: CHECKPOINT_METRICS is empty")
    for key, value in metrics.items():
        if not np.isfinite(value):
            raise AssertionError(f"TrialExamples: metric '{key}' is not finite: {value}")
    for key, (lo, hi) in limits.items():
        if key not in metrics:
            raise AssertionError(f"TrialExamples: missing checkpoint metric '{key}'")
        value = float(metrics[key])
        if value < float(lo) or value > float(hi):
            raise AssertionError(
                f"TrialExamples: metric '{key}'={value:.6f} outside [{float(lo):.6f}, {float(hi):.6f}]"
            )
    print(f"Numeric checkpoints for {topic}:", metrics)


In [ ]:
# TrialExamples: build a trial from spikes, covariates, events, and history.
from nstat.compat.matlab import Covariate, CovColl, Events, History, Trial, nspikeTrain, nstColl

length_trial = 1.0
window_times = np.array([0.0, 0.1, 0.2, 0.4], dtype=float)
h = History(bin_edges_s=window_times)

t = np.arange(0.0, length_trial + 0.001, 0.001)
position = Covariate(
    time=t,
    data=np.column_stack([np.cos(2.0 * np.pi * t), np.sin(2.0 * np.pi * t)]),
    name="Position",
    labels=["x", "y"],
)
force = Covariate(
    time=t,
    data=np.column_stack([np.sin(2.0 * np.pi * 4.0 * t), np.cos(2.0 * np.pi * 4.0 * t)]),
    name="Force",
    labels=["f_x", "f_y"],
)
cc = CovColl([position, force])
cc.setMaxTime(length_trial)

e_times = np.sort(rng.random(2) * length_trial)
e = Events(times=e_times, labels=["E_1", "E_2"])

trains = []
for i in range(4):
    spk = np.sort(rng.random(100) * length_trial)
    trains.append(nspikeTrain(spike_times=spk, t_start=0.0, t_end=length_trial, name=f"n{i+1}"))
spikeColl = nstColl(trains)

trial1 = Trial(spikes=spikeColl, covariates=cc)
trial1.setTrialEvents(e)
trial1.setHistory(h)

fig, axes = plt.subplots(2, 2, figsize=(10.0, 7.2))
plt.sca(axes[0, 0])
h.plot()
axes[0, 0].set_title("History windows")
plt.sca(axes[0, 1])
cc.plot()
axes[0, 1].set_title("Covariates")
plt.sca(axes[1, 0])
e.plot()
axes[1, 0].set_title("Events")
plt.sca(axes[1, 1])
spikeColl.plot()
axes[1, 1].set_title("Spike raster")
for ax in axes.ravel():
    ax.set_xlabel("time [s]")
plt.tight_layout()
plt.show()

trial1.setCovMask(["Position", "Force"])
hist_rows = trial1.getHistForNeurons([1, 2], binSize_s=0.01)

fig2 = plt.figure(figsize=(8.0, 3.8))
if hist_rows:
    plt.imshow(hist_rows[0].T, aspect="auto", origin="lower", cmap="magma")
    plt.title("Neuron 1 history matrix")
    plt.xlabel("time-bin index")
    plt.ylabel("history basis")
    plt.colorbar(fraction=0.04, pad=0.02)
else:
    plt.plot([], [])
plt.tight_layout()
plt.show()

assert len(hist_rows) >= 1
assert hist_rows[0].shape[1] == h.getNumBins()
history = h
spikes = spikeColl.getNST(0)
H = history.computeHistory(spikes.spike_times, t)
assert H.ndim == 2 and H.shape[1] == history.n_bins
assert spikes.spike_times.size > 5

CHECKPOINT_METRICS = {
    "history_bins": float(h.getNumBins()),
    "hist_rows_neuron1": float(hist_rows[0].shape[0] if hist_rows else 0.0),
}
CHECKPOINT_LIMITS = {
    "history_bins": (3.0, 3.0),
    "hist_rows_neuron1": (50.0, 2000.0),
}


In [ ]:
# Execution checkpoints used by CI.
assert TOPIC != "", "Missing topic metadata"
validate_numeric_checkpoints(CHECKPOINT_METRICS, CHECKPOINT_LIMITS, TOPIC)
print("Topic-specific checkpoint for", TOPIC)
print("Notebook checkpoints passed for", TOPIC)


## Next steps

- Compare this notebook with the corresponding MATLAB helpfile output in the validation PDF.
- Use `tools/reports/generate_validation_pdf.py` to run side-by-side parity scoring.
- Refine model assumptions for this specific example until parity thresholds pass.
